In [2]:
import pandas as pd 

Load the omdb raw data

In [3]:
df = pd.read_csv("/Users/ishik/Desktop/brooklyn99-sql-analytics/data/omdb_raw.csv")
df.head()

,Title,Year,Rated,Released,Season,Episode,Runtime,Genre,Director,Writer,...,Awards,Poster,Ratings,Metascore,imdbRating,imdbVotes,imdbID,seriesID,Type,Response
0,Pilot,2013,TV-14,17 Sep 2013,1,1,23 min,"Comedy, Crime","Phil Lord, Christopher Miller","Dan Goor, Michael Schur",...,NaN,https://m.media-amazon.com/images/M/MV5BMTYwOD...,"[{'Source': 'Internet Movie Database', 'Value'...",NaN,7.7,6535,tt2936284,tt2467372,episode,True
1,The Tagger,2013,TV-14,24 Sep 2013,1,2,21 min,"Comedy, Crime",Craig Zisk,"Dan Goor, Michael Schur, Norm Hiscock",...,NaN,https://m.media-amazon.com/images/M/MV5BMTYzNT...,"[{'Source': 'Internet Movie Database', 'Value'...",NaN,7.5,5147,tt3179592,tt2467372,episode,True
2,The Slump,2013,TV-14,01 Oct 2013,1,3,21 min,"Comedy, Crime",Julie Anne Robinson,"Dan Goor, Michael Schur, Prentice Penny",...,NaN,https://m.media-amazon.com/images/M/MV5BMjAwMz...,"[{'Source': 'Internet Movie Database', 'Value'...",NaN,7.5,4796,tt3211762,tt2467372,episode,True
3,M.E. Time,2013,TV-14,08 Oct 2013,1,4,21 min,"Comedy, Crime",Troy Miller,"Dan Goor, Michael Schur, Gil Ozeri",...,NaN,https://m.media-amazon.com/images/M/MV5BZjYzMz...,"[{'Source': 'Internet Movie Database', 'Value'...",NaN,7.6,4665,tt3220646,tt2467372,episode,True
4,The Vulture,2013,TV-14,15 Oct 2013,1,5,22 min,"Comedy, Crime",Jason Ensler,"Dan Goor, Michael Schur, Laura McCreary",...,NaN,https://m.media-amazon.com/images/M/MV5BMTIwMm...,"[{'Source': 'Internet Movie Database', 'Value'...",NaN,7.9,4723,tt3113610,tt2467372,episode,True


Dropping unnecessary columns

In [4]:
df = df.drop(columns=['Year', 'Rated','Released','Runtime','Genre','Awards','Poster','Ratings','Metascore','imdbRating','imdbVotes','imdbID','seriesID','Type','Response','Plot','Language','Country','Actors'])
df.head()


,Title,Season,Episode,Director,Writer
0,Pilot,1,1,"Phil Lord, Christopher Miller","Dan Goor, Michael Schur"
1,The Tagger,1,2,Craig Zisk,"Dan Goor, Michael Schur, Norm Hiscock"
2,The Slump,1,3,Julie Anne Robinson,"Dan Goor, Michael Schur, Prentice Penny"
3,M.E. Time,1,4,Troy Miller,"Dan Goor, Michael Schur, Gil Ozeri"
4,The Vulture,1,5,Jason Ensler,"Dan Goor, Michael Schur, Laura McCreary"


Check number of nulls in each column

In [5]:
df.shape

(153, 5)

In [6]:
df.isna().sum()

Title       0
Season      0
Episode     0
Director    3
Writer      3
dtype: int64

There three episodes where the writer and director is missing

In [7]:
df[df["Director"].isna()]

,Title,Season,Episode,Director,Writer
9,Thanksgiving,1,10,NaN,NaN
86,Your Honor,4,19,NaN,NaN
130,Manhunter,7,1,NaN,NaN


As the director and writer is missing for the same three episode. I have decided to use the IMDB website and manually input the writers and directors. For only 3 missing values I plan on skipping the use of another website to pull the values.

In [8]:
df.loc[(df["Season"] == 1) & (df["Episode"] == 10),"Director"] = "Jorma Taccone"
df.loc[(df["Season"]==1) & (df["Episode"] == 10),"Writer"] = "Luke Del Tredici, Lesley Arfin, Gil Ozeri"

In [9]:
df.loc[(df["Season"] == 4) & (df["Episode"] == 19),"Director"] = "Michael McDonald"
df.loc[(df["Season"] == 4) & (df["Episode"] == 19),"Writer"] = "David Phillips, Carly Hallam, Justin Noble"

In [10]:
df.loc[(df["Season"] == 7) & (df["Episode"] == 1),"Director"] = "Cortney Carrillo"
df.loc[(df["Season"] == 7) & (df["Episode"] == 1),"Writer"] = "David Phillips, Dewayne Perkins, Vanessa Ramos"

In [11]:
df[df["Director"].isna()]

,Title,Season,Episode,Director,Writer


split the director and writer columns so that each person gets their own column

In [12]:
split_df = df['Director'].str.split(',', expand=True)

split_df.columns = [f'Director_{i}' for i in split_df.columns]

result = pd.concat([df, split_df], axis=1)

In [13]:
result.head()

,Title,Season,Episode,Director,Writer,Director_0,Director_1
0,Pilot,1,1,"Phil Lord, Christopher Miller","Dan Goor, Michael Schur",Phil Lord,Christopher Miller
1,The Tagger,1,2,Craig Zisk,"Dan Goor, Michael Schur, Norm Hiscock",Craig Zisk,None
2,The Slump,1,3,Julie Anne Robinson,"Dan Goor, Michael Schur, Prentice Penny",Julie Anne Robinson,None
3,M.E. Time,1,4,Troy Miller,"Dan Goor, Michael Schur, Gil Ozeri",Troy Miller,None
4,The Vulture,1,5,Jason Ensler,"Dan Goor, Michael Schur, Laura McCreary",Jason Ensler,None


In [14]:
split_df1 = result['Writer'].str.split(',', expand=True)

split_df1.columns = [f'Writer_{i}' for i in split_df1.columns]

result_final = pd.concat([result, split_df1], axis=1)

In [15]:
result_final.head()

,Title,Season,Episode,Director,Writer,Director_0,Director_1,Writer_0,Writer_1,Writer_2
0,Pilot,1,1,"Phil Lord, Christopher Miller","Dan Goor, Michael Schur",Phil Lord,Christopher Miller,Dan Goor,Michael Schur,None
1,The Tagger,1,2,Craig Zisk,"Dan Goor, Michael Schur, Norm Hiscock",Craig Zisk,None,Dan Goor,Michael Schur,Norm Hiscock
2,The Slump,1,3,Julie Anne Robinson,"Dan Goor, Michael Schur, Prentice Penny",Julie Anne Robinson,None,Dan Goor,Michael Schur,Prentice Penny
3,M.E. Time,1,4,Troy Miller,"Dan Goor, Michael Schur, Gil Ozeri",Troy Miller,None,Dan Goor,Michael Schur,Gil Ozeri
4,The Vulture,1,5,Jason Ensler,"Dan Goor, Michael Schur, Laura McCreary",Jason Ensler,None,Dan Goor,Michael Schur,Laura McCreary


In [16]:
result_cleaned = result_final.drop(columns=["Director","Writer"])
result_cleaned.head()

,Title,Season,Episode,Director_0,Director_1,Writer_0,Writer_1,Writer_2
0,Pilot,1,1,Phil Lord,Christopher Miller,Dan Goor,Michael Schur,None
1,The Tagger,1,2,Craig Zisk,None,Dan Goor,Michael Schur,Norm Hiscock
2,The Slump,1,3,Julie Anne Robinson,None,Dan Goor,Michael Schur,Prentice Penny
3,M.E. Time,1,4,Troy Miller,None,Dan Goor,Michael Schur,Gil Ozeri
4,The Vulture,1,5,Jason Ensler,None,Dan Goor,Michael Schur,Laura McCreary


In [18]:
df_main = pd.read_csv("/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_episodes_cleaned.csv")
df_main.head()

,Season,Episode,Title,Airdate,Rating,Total Votes,Episode Description,episode_category,is_holiday
0,1,1,Pilot,2014-01-16 00:00:00,7.8,4701,Detective Jake Peralta finds his work scrutini...,Great,Non-Holiday
1,1,2,The Tagger,2014-01-23 00:00:00,7.5,3833,"When Jake arrives late for work, Captain Holt ...",Great,Non-Holiday
2,1,3,The Slump,2014-01-30 00:00:00,7.6,3591,"With a backlog of unsolved cases, Jake finds h...",Great,Non-Holiday
3,1,4,M.E. Time,2014-02-06 00:00:00,7.7,3473,"Jake meets an attractive Medical Examiner, but...",Great,Non-Holiday
4,1,5,The Vulture,2014-02-13 00:00:00,8.0,3370,A detective from Major Crimes takes over Jake'...,Great,Non-Holiday


Merge the two datasets

In [19]:
merged_df = pd.merge(df_main,result_cleaned, on=["Season","Episode"], how="left")
merged_df.head()

,Season,Episode,Title_x,Airdate,Rating,Total Votes,Episode Description,episode_category,is_holiday,Title_y,Director_0,Director_1,Writer_0,Writer_1,Writer_2
0,1,1,Pilot,2014-01-16 00:00:00,7.8,4701,Detective Jake Peralta finds his work scrutini...,Great,Non-Holiday,Pilot,Phil Lord,Christopher Miller,Dan Goor,Michael Schur,None
1,1,2,The Tagger,2014-01-23 00:00:00,7.5,3833,"When Jake arrives late for work, Captain Holt ...",Great,Non-Holiday,The Tagger,Craig Zisk,None,Dan Goor,Michael Schur,Norm Hiscock
2,1,3,The Slump,2014-01-30 00:00:00,7.6,3591,"With a backlog of unsolved cases, Jake finds h...",Great,Non-Holiday,The Slump,Julie Anne Robinson,None,Dan Goor,Michael Schur,Prentice Penny
3,1,4,M.E. Time,2014-02-06 00:00:00,7.7,3473,"Jake meets an attractive Medical Examiner, but...",Great,Non-Holiday,M.E. Time,Troy Miller,None,Dan Goor,Michael Schur,Gil Ozeri
4,1,5,The Vulture,2014-02-13 00:00:00,8.0,3370,A detective from Major Crimes takes over Jake'...,Great,Non-Holiday,The Vulture,Jason Ensler,None,Dan Goor,Michael Schur,Laura McCreary


Check the merged dataframe

In [20]:
merged_df.shape

(153, 15)

In [24]:
merged_df.rename(columns={"Title_x": "Title"}, inplace=True)

In [26]:
merged_df = merged_df.drop(columns=["Title_y"])

In [27]:
merged_df.head()

,Season,Episode,Title,Airdate,Rating,Total Votes,Episode Description,episode_category,is_holiday,Director_0,Director_1,Writer_0,Writer_1,Writer_2
0,1,1,Pilot,2014-01-16 00:00:00,7.8,4701,Detective Jake Peralta finds his work scrutini...,Great,Non-Holiday,Phil Lord,Christopher Miller,Dan Goor,Michael Schur,None
1,1,2,The Tagger,2014-01-23 00:00:00,7.5,3833,"When Jake arrives late for work, Captain Holt ...",Great,Non-Holiday,Craig Zisk,None,Dan Goor,Michael Schur,Norm Hiscock
2,1,3,The Slump,2014-01-30 00:00:00,7.6,3591,"With a backlog of unsolved cases, Jake finds h...",Great,Non-Holiday,Julie Anne Robinson,None,Dan Goor,Michael Schur,Prentice Penny
3,1,4,M.E. Time,2014-02-06 00:00:00,7.7,3473,"Jake meets an attractive Medical Examiner, but...",Great,Non-Holiday,Troy Miller,None,Dan Goor,Michael Schur,Gil Ozeri
4,1,5,The Vulture,2014-02-13 00:00:00,8.0,3370,A detective from Major Crimes takes over Jake'...,Great,Non-Holiday,Jason Ensler,None,Dan Goor,Michael Schur,Laura McCreary


Turn this dataframe into csv

In [29]:
merged_df.to_csv("/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_episodes_with_writer_director.csv", index=False)